In [ ]:
%load_ext autoreload
%autoreload 2
import os,sys
import numpy as np
import pandas as pd
import scanpy as sc

import pseudodynamics as pdp

os.chdir(pdp.main_dir)
import scripts.fate_eval_pipeline as fate_eval


In [ ]:
adata = sc.read_h5ad("data/klein_addpop.h5ad")

F_obs = pd.read_csv("data/klein/F_obs.csv", index_col=0)
F_obs = F_obs[F_obs.index.astype(str).isin(adata.obs_names)]

In [8]:
# read config
config_path = "logs/klein_DM_10_lD1_lv1_lgNone/pde_params_tsense/V0_config.json"
config = pdp.ExperimentConfig(config_path)

# get key from config
cellstate_key = config.dataset_config.get("cellstate_key", None)
timepoint_key = config.dataset_config.get("timepoint_key", "timepoint_tx_days")

In [ ]:
pde_model = pdp.models.pde_params.load_from_checkpoint(
    config.find_lastest_ckpt()
)

In [ ]:
full_DS = pdp.reader.TwoTimpepoint_AnnDS(adata, 
                                         split=None,
                                         **config.dataset_config
                                         )


bifate_mask = np.load("data/klein/neutrophil_monocyte_trajectory_mask.npy", allow_pickle=True)
clearn = sc.read_h5ad("data/weineb_invitro.h5ad")
bifate_obs = clearn[bifate_mask].obs_names
adata.obs['bifate_mask'] = adata.obs_names.isin(bifate_obs)

try :
    full_DS = pdp.reader.TwoTimpepoint_AnnDS(adata, 
                                         split=None,
                                         **config.dataset_config
                                         )
    adata.obsm[f'guassian_kde_u_{cellstate_key}'] = full_DS.u_b.T.numpy()
except:
    print("config is not provided, Dataset not defined")


Dataset : Computing density :
	 `density_funs` not specified, default estimator gaussian kde
Dataset : all cells are used


In [ ]:
# adata.write_h5ad('data/klein_addpop.h5ad')

# clone analysis

In [9]:
tp_values  = sorted(adata.obs[timepoint_key].unique())
tp_start   = tp_values[0]

In [107]:
fobs_idx_str  = F_obs.index.astype(str)

valid_mask  = adata.obs_names.astype(str).isin(fobs_idx_str)
tp_mask     = (adata.obs[timepoint_key] == int(tp_start)).values
start_mask  = valid_mask & tp_mask

In [ ]:
valid_mask.shape

In [97]:
clearn

AnnData object with n_obs × n_vars = 130887 × 2447
    obs: 'Library', 'Cell barcode', 'Time point', 'Starting population', 'Cell type annotation', 'Well', 'SPRING-x', 'SPRING-y', 'clone_idx', 'fate_observed', 't0_fated', 'train', 'test'
    uns: 'Cell type annotation_colors', 'neighbors', 'train_colors', 'umap'
    obsm: 'X_pca', 'X_spring', 'X_umap', 'X_umap_old'
    layers: 'X_scaled'
    obsp: 'connectivities', 'distances'

In [95]:
print(clearn[F_obs.index].obs['clone_idx'].nunique())
clone_idxs = clearn[F_obs.index].obs['clone_idx'].astype(int).unique()

1407


In [99]:
clearn_clone_sum = clearn.obs.groupby('clone_idx').agg({"Time point":"nunique", 'Starting population':'count', 'Well':'unique'})

In [101]:
clearn_clone_sum.loc[clone_idxs]

,Time point,Starting population,Well
clone_idx,,,
1626.0,3,17,"[1, 2, 0]"
5039.0,2,5,"[1, 0]"
4810.0,2,3,"[0, 2]"
861.0,2,3,"[2, 0]"
1161.0,2,2,"[0, 1]"
...,...,...,...
2973.0,3,4,"[1, 0]"
1730.0,3,3,"[1, 0]"
4419.0,3,26,"[2, 0]"


In [110]:
adata.uns['eval_clones'] = [f"Clone_{int(c)}" for c in clone_idxs]

In [112]:
clone_sum.loc[adata.uns['eval_clones'] ]

,Time_point,Population,Well
clones,,,
Clone_1626,3,17,"[1, 2, 0]"
Clone_5039,2,4,"[1, 0]"
Clone_4810,2,3,"[0, 2]"
Clone_861,2,3,"[2, 0]"
Clone_1161,2,2,"[0, 1]"
...,...,...,...
Clone_2973,3,4,"[1, 0]"
Clone_1730,3,3,"[1, 0]"
Clone_4419,3,26,"[2, 0]"


In [115]:
adata

AnnData object with n_obs × n_vars = 126861 × 5000
    obs: 'Time_point', 'Population', 'Annotation', 'Well', 'time_cat', 'leiden', 'comb', 'label_man', 'clones', 'Meta clones', 'group', 'timepoint_tx_days', 'selected_clonal_cells', 'palantir_pseudotime', 'palantir_entropy', 'split', 'bifate_mask'
    var: 'symbol', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'Annotation_colors', 'DM_EigenValues', 'Meta clones_colors', 'Population_colors', 'Time_point_colors', 'comb_colors', 'diffmap_evals', 'group_colors', 'hvg', 'iroot', 'label_man_colors', 'label_man_sizes', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'paga', 'palantir_waypoints', 'pca', 'pop', 'root_cb', 'time_cat_colors', 'umap', 'eval_clones'
    obsm: 'DM_EigenVectors', 'DM_EigenVectors_multiscaled', 'X_diffmap', 'X_pca', 'X_pca_scaled', 'X_umap', 'delta_DM', 'delta_PC', 'palantir_fate_probabilities', 'guassian_kde_u_DM_EigenVectors'
    varm: 'PCs'
    obsp: 'DM_Kernel', 'DM_Simila

In [106]:
clearn[F_obs.index].obs['clone_idx']

13199    1626.0
13210    5039.0
13226    4810.0
13235     861.0
13256    1161.0
          ...  
69313    1950.0
69317    5801.0
69322     352.0
69324    5663.0
69326    4897.0
Name: clone_idx, Length: 2081, dtype: float64

In [79]:
#  clone filtering
clone_sum = adata.obs.groupby('clones').agg({"Time_point":"nunique", "Population":'count', 'Well':'unique'})


In [ ]:
[clone_idxs]

array([1626, 5039, 4810, ..., 4419, 3016, 5663])

In [133]:
count_table = adata.obs[['clones','split', 'Time_point', 'Annotation']].pivot_table(
    index="clones", 
    columns='split', 
    aggfunc='count',
    fill_value=0
)
count_table

Annotation              Time_point             
split            test  train   val       test  train   val
clones                                                    
Clone_0             0      5     0          0      5     0
Clone_1            15      3     2         15      3     2
Clone_2            13      2     0         13      2     0
Clone_3             0      2     0          0      2     0
Clone_4             0      7     2          0      7     2
...               ...    ...   ...        ...    ...   ...
Clone_5860          2      0     0          2      0     0
Clone_5861          2      0     0          2      0     0
Clone_5862          1      1     0          1      1     0
Clone_5863          4      0     0          4      0     0
Clone_nan       29945  44061  4876      29945  44061  4876

[5860 rows x 6 columns]

In [ ]:
proportion_table = pd.crosstab(
    adata.obs['clones'], 
    adata.obs['split'], 
    normalize='index'  # This makes each row sum to 1c
)

proportion_table.query("`test` == 1")

split,test,train,val
clones,,,
Clone_0,0.000000,1.000000,0.000000
Clone_1,0.750000,0.150000,0.100000
Clone_2,0.866667,0.133333,0.000000
Clone_3,0.000000,1.000000,0.000000
Clone_4,0.000000,0.777778,0.222222
...,...,...,...
Clone_5860,1.000000,0.000000,0.000000
Clone_5861,1.000000,0.000000,0.000000
Clone_5862,0.500000,0.500000,0.000000


In [135]:
proportion_table.query("`test` == 1")

split,test,train,val
clones,,,
Clone_7,1.0,0.0,0.0
Clone_14,1.0,0.0,0.0
Clone_16,1.0,0.0,0.0
Clone_17,1.0,0.0,0.0
Clone_18,1.0,0.0,0.0
...,...,...,...
Clone_5846,1.0,0.0,0.0
Clone_5851,1.0,0.0,0.0
Clone_5860,1.0,0.0,0.0


In [ ]:
proportion_table = pd.crosstab(
    adata.obs['clones'], 
    adata.obs['Time_point'], 
    normalize='index'  # This makes each row sum to 1c
)

proportion_table

# cell state simulation

In [139]:
n_dim = config.dataset_config['n_dimension']
start_cells = adata[valid_mask].obsm[cellstate_key][:,:n_dim]
start_cells.shape

(2031, 10)

In [ ]:
sim_fn = fate_eval.make_pseudodynamics_sim_fn(t_start_norm=0.0, t_end_norm=4.0)

array([ True,  True,  True, ...,  True,  True,  True])

In [147]:
ids_str  = np.array([str(i) for i in adata[valid_mask].obs_names])
fobs_str = F_obs.index.astype(str)
in_fobs  = pd.Series(ids_str).isin(fobs_str).values

In [159]:
config.experiment_config['save_dir'].split("/")[-2]

'klein_DM_10_lD1_lv1_lgNone'

In [148]:
ids_filt  = ids_str[in_fobs]

In [151]:
ids_filt.shape

(2031,)

In [152]:
ids_str.shape

(2031,)

In [154]:
F_obs.index

Index([13199, 13210, 13226, 13235, 13256, 13258, 13259, 13260, 13266, 13274,
       ...
       69297, 69299, 69301, 69308, 69312, 69313, 69317, 69322, 69324, 69326],
      dtype='int64', length=2031)

In [ ]:
adata_train = adata[adata.obs.Well != 2]
x_ref = adata_train.obsm[cellstate_key][:, :n_dim].astype(np.float32)
y_ref = adata_train.obs["Annotation"].values.astype(str)


result = fate_eval.run_fate_evaluation(start_cells, 
                                        list(adata[valid_mask].obs_names), 
                                        pde_model, 
                                        sim_fn,
                                        F_obs, x_ref, y_ref,
                                        cell_types=None, 
                                        n_sims=100, 
                                        k=20, 
                                        device='cuda:0')